<a href="https://colab.research.google.com/github/yeseul-huh/Editorial-Planning-Strategies-for-Serialized-Novels/blob/main/%EC%8B%9D%EB%AF%BC%EC%A7%80%EA%B8%B0_%EC%8B%A0%EB%AC%B8_%EC%97%B0%EC%9E%AC_%EC%86%8C%EC%84%A4%EC%9D%98_%EC%A7%80%EB%A9%B4_%EA%B8%B0%ED%9A%8D_%EC%A0%84%EB%9E%B5_%EC%97%B0%EA%B5%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 필수 라이브러리 및 한글 폰트 설치

In [ ]:
# 1. 필수 라이브러리 및 한글 폰트 설치
!pip install konlpy
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

# time interval 정보를 포함한 node, edge 파일 생성

In [ ]:
import pandas as pd
import re
from google.colab import files

# 1. 파일명 설정
FILE_NAME = ''

def generate_3way_network(file_path):
    try:
        df = pd.read_excel(file_path)
    except FileNotFoundError:
        return print("파일을 찾을 수 없습니다. CSV 파일을 업로드해주세요.")

    cols = df.columns.tolist()

    # 컬럼 매핑 (데이터셋의 실제 이름 확인)
    num_col = [c for c in cols if '일련번호' in c or '번호' in c or 'No' in c][0]
    author_col = '작가명(정제)' if '작가명(정제)' in cols else '작가명'
    title_col = '작품명(정제)' if '작품명(정제)' in cols else '작품명'
    ill_col = '삽화가(*추정)' if '삽화가(*추정)' in cols else '삽화가'
    date_col = '연재시작'
    media_col = '연재지면'

    # 2. 범위 제한: 일련번호 1~564번 (특수문예 포함)
    df_564 = df[df[num_col] <= 564].copy()

    # 3. 정제 함수 (역/번안 통합 및 *추정 제외)
    def clean_name(name):
        if pd.isna(name): return "미상"
        name = str(name).strip()
        if '*' in name: return None # 추정 데이터 제외

        name = re.sub(r'\(.*\)', '', name) # 괄호 제거
        name = re.sub(r'\(.*?\)', '', name)
        for suffix in ['역', '번안', '편']:
            if name.endswith(suffix): name = name[:-len(suffix)]
        return name.strip()

    # 4. 데이터 가공 및 분리 (/)
    df_564['작가_ID'] = df_564[author_col].apply(clean_name)
    df_564['신문사_ID'] = df_564[media_col].str.strip()

    # 삽화가 분리 (/) 및 정제
    def split_illustrator(val):
        if pd.isna(val) or '*' in str(val): return [None]
        return [clean_name(x) for x in str(val).split('/')]

    df_564['삽화가_리스트'] = df_564[ill_col].apply(split_illustrator)
    df_exploded = df_564.explode('삽화가_리스트')
    df_exploded = df_exploded.rename(columns={'삽화가_리스트': '삽화가_ID'})

    # 유효 데이터 필터링
    df_net = df_exploded[
        (df_exploded['작가_ID'].notna()) & (df_exploded['작가_ID'] != '미상') &
        (df_exploded['삽화가_ID'].notna()) & (df_exploded['삽화가_ID'] != '미상')
    ].copy()

    # 연도 정보 추출
    df_net['Year'] = df_net[date_col].apply(lambda x: int(str(x)[:4]) if pd.notna(x) else None)

    # 5. Edges 생성 (3가지 관계)
    def make_edge(df, src, tgt, etype):
        return pd.DataFrame({
            'Source': df[src], 'Target': df[tgt], 'Year': df['Year'],
            'Newspaper': df[media_col], 'Work_Title': df[title_col], 'Edge_Type': etype
        })

    e1 = make_edge(df_net, '작가_ID', '삽화가_ID', 'Author-Illustrator')
    e2 = make_edge(df_net, '작가_ID', '신문사_ID', 'Author-Newspaper')
    e3 = make_edge(df_net, '신문사_ID', '삽화가_ID', 'Newspaper-Illustrator')

    edges_final = pd.concat([e1, e2, e3], ignore_index=True)

    # 6. Nodes 생성
    n1 = pd.DataFrame({'ID': df_net['작가_ID'], 'Year': df_net['Year'], 'Type': 'Author'})
    n2 = pd.DataFrame({'ID': df_net['삽화가_ID'], 'Year': df_net['Year'], 'Type': 'Illustrator'})
    n3 = pd.DataFrame({'ID': df_net['신문사_ID'], 'Year': df_net['Year'], 'Type': 'Newspaper'})

    nodes_final = pd.concat([n1, n2, n3], ignore_index=True).drop_duplicates('ID')
    nodes_final['Label'] = nodes_final['ID']

    # 7. 파일 저장 및 다운로드
    nodes_final.to_csv('nodes_3way_refined.csv', index=False, encoding='utf-8-sig')
    edges_final.to_csv('edges_3way_refined.csv', index=False, encoding='utf-8-sig')

    print(f"생성 완료! 노드: {len(nodes_final)}, 에지: {len(edges_final)}")
    files.download('nodes_3way_refined.csv')
    files.download('edges_3way_refined.csv')

# 실행
generate_3way_network(FILE_NAME)

# 3. 데이터 분석 및 해석

## 3-1. [형식] 지면의 호흡: 단일 장편의 지속인가, 다각적 단편의 분할인가

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. 데이터 로드
file_path = ''
df = pd.read_excel(file_path)

# 2. 전처리 및 분류 로직
# 시대 구분 (1930-40년대 통합)
def classify_era(date_str):
    try:
        year = int(str(date_str)[:4])
        if year < 1930: return '1920년대'
        elif 1930 <= year <= 1945: return '1930-40년대'
    except: return None
    return None

df['시대'] = df['연재시작'].apply(classify_era)

# 특수 문예 키워드 정의
special_keywords = ['당선소설', '독자문단', '독자문예', '부인문예', '콩쿨 입선소설', '현상문예']

def categorize_novel(row):
    # 특수 문예 및 특집 구분 컬럼을 먼저 확인
    spec_val = str(row['특수 문예 및 특집 구분'])
    if any(k in spec_val for k in special_keywords):
        return '특수 문예'

    # 그 외 일반 소설은 분량에 따라 구분
    length_val = str(row['분량에 따른 구분'])
    if '단편' in length_val: return '단편'
    if '중편' in length_val: return '중편'
    if '장편' in length_val: return '장편'
    return None

df['소설구분'] = df.apply(categorize_novel, axis=1)

# 유효 데이터 필터링
df_clean = df[df['시대'].notna() & df['소설구분'].notna()].copy()

# 3. 시각화 함수
def plot_novel_trends(target_df, title, filename):
    ct = pd.crosstab(target_df['시대'], target_df['소설구분'])

    # 순서 고정 (단->중->장->특수)
    order = ['단편', '중편', '장편', '특수 문예']
    order = [c for c in order if c in ct.columns]
    ct = ct[order]

    ct_ratio = ct.div(ct.sum(axis=1), axis=0) * 100

    plt.rc('font', family='NanumGothic')
    fig, ax = plt.subplots(figsize=(12, 8), dpi=300)

    # 색상 설정 (특수 문예는 눈에 띄는 주황 계열로 설정)
    colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99']
    ct_ratio.plot(kind='bar', stacked=True, ax=ax, color=colors, width=0.6)

    plt.title(title, fontsize=16, pad=20, fontweight='bold')
    plt.ylabel('비율 (%)', fontsize=12)
    plt.xlabel('시대', fontsize=12)
    plt.xticks(rotation=0)
    plt.legend(title='소설 구분', bbox_to_anchor=(1.02, 1), loc='upper left')

    # 수치 표시
    for i, era in enumerate(ct.index):
        cumulative_height = 0
        for cat in order:
            count = ct.loc[era, cat]
            percentage = ct_ratio.loc[era, cat]
            if count > 0:
                label = f'{count}편\n({percentage:.1f}%)'
                y_pos = cumulative_height + (percentage / 2)
                ax.text(i, y_pos, label, ha='center', va='center',
                        fontsize=10, fontweight='bold', color='black')
            cumulative_height += percentage

    plt.tight_layout()
    plt.savefig(filename, bbox_inches='tight')
    plt.show()

# 4. 분석 실행
# 조선일보
plot_novel_trends(df_clean[df_clean['연재지면'] == '조선일보'],
                  '조선일보: 시대별 소설 구분 변화 (특수 문예 포함)', 'chosun_with_special.png')

# 동아일보
plot_novel_trends(df_clean[df_clean['연재지면'] == '동아일보'],
                  '동아일보: 시대별 소설 구분 변화 (특수 문예 포함)', 'donga_with_special.png')

##3-2. [인적 네트워크] 지면의 파트너십: 누구의 문장으로 채우고, 누구의 선으로 시각화했는가

### 작품을 많이 실었던 작가 상위 10명

In [ ]:
import pandas as pd
import re

# 1. 파일명 설정
FILE_NAME = ''

def colab_final_integrated_analysis(file_path):
    try:
        df = pd.read_excel(file_path)
    except FileNotFoundError:
        return "파일을 찾을 수 없습니다. CSV 파일을 업로드해주세요."

    # 2. 범위 제한: 일련번호 1~564번
    num_col = [col for col in df.columns if col in ['일련번호', '번호', 'No', 'no']]
    df_564 = df[df[num_col[0]] <= 564].copy() if num_col else df.iloc[:564].copy()

    # 3. 작가명 정제 (역/번안 통합 및 괄호 제거)
    def clean_author(name):
        if pd.isna(name): return "미상"
        name = str(name).strip()
        name = re.sub(r'\(.*\)', '', name)
        name = re.sub(r'\(.*?\)', '', name)
        for suffix in ['역', '번안', '편']:
            if name.endswith(suffix): name = name[:-len(suffix)]
        return name.strip()

    df_564['작가명_최종정제'] = df_564['작가명(정제)'].apply(clean_author)

    # 4. [순위 분석] 특수문예 포함 다작 작가 순위
    top_authors = {}
    for media in df_564['연재지면'].unique():
        subset = df_564[df_564['연재지면'] == media]
        top_authors[media] = subset[subset['작가명_최종정제'] != '미상']['작가명_최종정제'].value_counts().head(10)

    # 5. [지표 분석] 특수문예 제외 작가 다양성 통계
    special_keywords = ['당선소설', '독자문단', '독자문예', '부인문예', '콩쿨 입선소설', '현상문예']
    df_normal = df_564[~df_564['특수 문예 및 특집 구분'].str.contains('|'.join(special_keywords), na=False)].copy()

    def get_stats(group):
        total, unique = len(group), group['작가명_최종정제'].nunique()
        one_time = sum(group['작가명_최종정제'].value_counts() == 1)
        return pd.Series({
            '전체 작품 수': int(total), '고유 작가 수': int(unique),
            '작가 1인당 평균 작품 수': round(total / unique, 2),
            '단 1회 작가 비중 (%)': round((one_time / unique) * 100, 1)
        })

    summary = df_normal.groupby('연재지면').apply(get_stats).reset_index()
    return summary, top_authors

# 분석 실행
summary, top_list = colab_final_integrated_analysis(FILE_NAME)

# 결과 출력
print("--- [1. 작가 다양성 지표 (특수문예 제외, 역/번안 통합)] ---")
print(summary.to_string(index=False))

print("\n--- [2. 다작 작가 Top 10 (특수문예 포함, 역/번안 통합)] ---")
for media, tops in top_list.items():
    print(f"\n[{media}]")
    print(tops)

### 작가-삽화가 협업 리스트

In [ ]:
import pandas as pd

# 1. 데이터 로드
df = pd.read_excel('')

# 2. 필터링: 추정(*) 데이터 및 결측치 제외
col_illustrator = '삽화가(*추정)'
col_author = '작가명(정제)'
df_clean = df[df[col_illustrator].notna() & (df[col_illustrator] != '-')].copy()
df_clean = df_clean[~df_clean[col_illustrator].str.contains('\*', na=False)]

# 3. 작가명 분리 (Explode): '/' 구분자 기준
df_clean[col_illustrator] = df_clean[col_illustrator].str.split('/')
df_expanded = df_clean.explode(col_illustrator)
df_expanded[col_illustrator] = df_expanded[col_illustrator].str.strip() # 공백 제거

# 4. 매체명 간소화 및 집계
def simplify_media(name):
    return '조선' if '조선' in str(name) else '동아' if '동아' in str(name) else str(name)

df_expanded['매체_간소'] = df_expanded['연재지면'].apply(simplify_media)

def get_media_summary(series):
    counts = series.value_counts()
    return ", ".join([f"{m}({c})" for m, c in counts.items()])

# 5. 최종 집계
summary = df_expanded.groupby(col_illustrator).agg(
    협업_작가_수=(col_author, 'nunique'),
    활동매체_편수=('매체_간소', get_media_summary),
    총_작업_편수=(col_illustrator, 'count')
).reset_index()

# 6. 순위 및 컬럼 정리
summary.columns = ['삽화가명', '협업 작가 수', '활동매체(편수)', '총 작업 편수']
summary = summary.sort_values(by='총 작업 편수', ascending=False).reset_index(drop=True)
summary.insert(0, '순위', summary.index + 1)

summary.to_csv('final_illustrator_ranking_split.csv', index=False, encoding='utf-8-sig')


## 3-3. [홍보의 언어] 지면의 수사학: 어떤 언어와 표상으로 작품의 가치를 구축했는가

### 연재 예고 텍스트 분석

In [ ]:
import pandas as pd
import re  # 정규표현식 라이브러리 추가
from sklearn.feature_extraction.text import TfidfVectorizer
from konlpy.tag import Okt

# 1. 데이터 로드
df = pd.read_excel('')

# 2. 시대/매체 그룹화
def get_era(d):
    try:
        y = int(str(d)[:4])
        return '1920s' if 1920 <= y <= 1929 else '1930-40s' if 1930 <= y <= 1945 else None
    except: return None

df['Era'] = df['기사 날짜'].apply(get_era)
df['Media'] = df['연재지면'].apply(lambda x: '조선' if '조선' in str(x) else '동아' if '동아' in str(x) else None)
df['Group'] = df['Era'].astype(str) + '_' + df['Media'].astype(str)

# 3. [핵심] 괄호 제거 및 명사 추출 함수
okt = Okt()

def extract_pure_nouns(text):
    if pd.isna(text) or text == '-': return ""

    # [추가] < > 로 묶여 있는 텍스트 및 괄호 삭제
    clean_text = re.sub(r'<[^>]*>', ' ', str(text))

    # 형태소 분석을 통해 품사가 'Noun'인 것만 리스트로 반환 후 결합
    nouns = okt.nouns(clean_text)

    # 한 글자 명사 및 불용어 처리가 필요하다면 여기에 추가 (예: n for n in nouns if len(n) > 1)
    return " ".join(nouns)

# 4. 시대_매체 그룹별 명사 코퍼스 생성
groups = sorted([g for g in df['Group'].unique() if 'None' not in g])
corpus = []
valid_groups = []

for group in groups:
    texts = df[df['Group'] == group]['연재 예고 텍스트(정제)'].dropna()
    if not texts.empty:
        # 괄호 제거 및 명사 추출 적용
        noun_text = " ".join(texts.apply(extract_pure_nouns))
        corpus.append(noun_text)
        valid_groups.append(group)

# 5. TF-IDF 계산
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)
terms = vectorizer.get_feature_names_out()

# 6. 결과 저장
result_data = []
for i, group in enumerate(valid_groups):
    scores = tfidf_matrix.getrow(i).toarray()[0]
    for idx, score in enumerate(scores):
        if score > 0:
            result_data.append({'시대_매체': group, '단어': terms[idx], 'TF-IDF_점수': score})

final_df = pd.DataFrame(result_data).sort_values(by=['시대_매체', 'TF-IDF_점수'], ascending=[True, False])

final_df.to_csv('refined_pure_nouns_tfidf_no_tags.csv', index=False, encoding='utf-8-sig')
final_df.to_excel('refined_pure_nouns_tfidf_no_tags.xlsx', index=False)

print("괄호 제거, 명사 추출 및 파일 저장이 완료되었습니다.")

###작가의 말 분석

In [ ]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from konlpy.tag import Okt

# 1. 데이터 로드
df = pd.read_excel('')

# 2. 시대/매체 그룹화
def get_era(d):
    try:
        y = int(str(d)[:4])
        return '1920s' if 1920 <= y <= 1929 else '1930-40s' if 1930 <= y <= 1945 else None
    except: return None

df['Era'] = df['기사 날짜'].apply(get_era)
df['Media'] = df['연재지면'].apply(lambda x: '조선' if '조선' in str(x) else '동아' if '동아' in str(x) else None)
df['Group'] = df['Era'].astype(str) + '_' + df['Media'].astype(str)

# 3. 작가의 말 전용 불용어 리스트 적용
author_stopwords = [
    '소설', '작품', '작자', '작가', '필자', '독자', '여러분', '이야기',]

okt = Okt()

def extract_author_nouns(text):
    if pd.isna(text) or text == '-': return ""

    # < > 태그 및 내부 내용 삭제
    clean_text = re.sub(r'<[^>]*>', ' ', str(text))

    # 명사 추출
    nouns = okt.nouns(clean_text)

    # 1. 불용어 리스트 제거
    # 2. 한 글자 명사 제거 (len > 1)
    refined = [n for n in nouns if n not in author_stopwords and len(n) > 1]
    return " ".join(refined)

# 4. '작가의 말' 컬럼 대상 코퍼스 생성
# (실제 데이터 시트의 컬럼명인 '작가의 말(정제)'를 사용합니다)
target_col = '작가(역자)의 말(정제)'

groups = sorted([g for g in df['Group'].unique() if 'None' not in g])
corpus = []
valid_groups = []

for group in groups:
    texts = df[df['Group'] == group][target_col].dropna()
    if not texts.empty:
        noun_text = " ".join(texts.apply(extract_author_nouns))
        if noun_text.strip():
            corpus.append(noun_text)
            valid_groups.append(group)

# 5. TF-IDF 분석
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus)
terms = vectorizer.get_feature_names_out()

# 6. 결과 저장
result_data = []
for i, group in enumerate(valid_groups):
    scores = tfidf_matrix.getrow(i).toarray()[0]
    top_indices = scores.argsort()[::-1]

    print(f"\n[{group}] 작가의 말 핵심 키워드 (불용어 제거 후)")
    count = 0
    for idx in top_indices:
        if scores[idx] > 0 and count < 10:
            print(f"- {terms[idx]}: {scores[idx]:.4f}")
            count += 1
        if scores[idx] > 0:
            result_data.append({'시대_매체': group, '단어': terms[idx], 'TF-IDF_점수': scores[idx]})

final_df = pd.DataFrame(result_data).sort_values(by=['시대_매체', 'TF-IDF_점수'], ascending=[True, False])
final_df.to_excel('author_words_tfidf_final.xlsx', index=False)

### 외부 변화와 작가의 말 변화(수양동우회 계열의 영향력 측정)

In [ ]:
import pandas as pd
from konlpy.tag import Okt
from sklearn.feature_extraction.text import TfidfVectorizer
import matplotlib.pyplot as plt
import os

# 1. 파일 로드
files = os.listdir('.')
data_file = [f for f in files if '신문연재소설' in f and f.endswith('.csv')][0]
df = pd.read_excel(data_file)
df.columns = [c.strip() for c in df.columns]

# 2. 7년 주기 구간 설정
def classify_7years(year):
    if 1920 <= year <= 1926:
        return '1기(1920-26)'
    elif 1927 <= year <= 1933:
        return '2기(1927-33)'
    elif 1934 <= year <= 1940:
        return '3기(1934-40)'
    return None

df['연도'] = pd.to_numeric(df['연재시작'].str[:4], errors='coerce')
df['구간_7년'] = df['연도'].apply(classify_7years)

# 3. 유효 데이터 필터링
col_talk = '작가(역자)의 말(정제)'
df_talks = df[df['구간_7년'].notna() & df[col_talk].notna() & (df[col_talk].astype(str).str.strip() != "-")].copy()

# 4. 불용어 리스트 및 형태소 분석기 설정
stop_words = ['소설', '작품', '작자', '작가', '필자', '독자', '여러분', '이야기']
okt = Okt()
def noun_tokenizer(text):
    return [n for n in okt.nouns(str(text)) if len(n) > 1 and n not in stop_words]

# 5. TF-IDF 분석 함수
def analyze_tfidf(subset):
    if len(subset) < 2: return pd.DataFrame()
    vectorizer = TfidfVectorizer(tokenizer=noun_tokenizer, max_features=100)
    tfidf_matrix = vectorizer.fit_transform(subset[col_talk])
    feature_names = vectorizer.get_feature_names_out()
    scores = tfidf_matrix.mean(axis=0).getA1()
    return pd.DataFrame({'단어': feature_names, '점수': scores}).sort_values(by='점수', ascending=False).head(10)

# 6. 매체별/7년구간별 분석 실행
results = []
for media in ['조선일보', '동아일보']:
    for period in ['1기(1920-26)', '2기(1927-33)', '3기(1934-40)']:
        subset = df_talks[(df_talks['연재지면'] == media) & (df_talks['구간_7년'] == period)]
        top_nouns = analyze_tfidf(subset)
        if not top_nouns.empty:
            top_nouns['매체'] = media
            top_nouns['구간'] = period
            results.append(top_nouns)

all_results = pd.concat(results)

# 7. 시각화 (3개 구간 비교)
plt.rcParams['font.family'] = 'NanumGothic'
fig, axes = plt.subplots(2, 3, figsize=(20, 12), sharex=True)

for i, media in enumerate(['조선일보', '동아일보']):
    for j, period in enumerate(['1기(1920-26)', '2기(1927-33)', '3기(1934-40)']):
        data = all_results[(all_results['매체'] == media) & (all_results['구간'] == period)]
        if not data.empty:
            axes[i, j].barh(data['단어'][::-1], data['점수'][::-1], color='steelblue' if media == '조선일보' else 'indianred')
            axes[i, j].set_title(f"{media} | {period}", fontsize=14, fontweight='bold')
            axes[i, j].grid(axis='x', alpha=0.3)

plt.suptitle('매체별 7년 주기 수사적 변천사 분석 (명사 중심 TF-IDF)', fontsize=22, y=1.02)
plt.tight_layout()
plt.savefig('tfidf_7year_transition.png')

# 결과 출력
print(all_results.pivot_table(index='단어', columns=['매체', '구간'], values='점수').fillna(0))
all_results.to_excel('tfidf_7year_results.xlsx', index=False)